In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
%matplotlib inline

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


In [2]:
# Load news dataset
news_df = pd.read_csv('../data/raw/raw_analyst_ratings.csv', index_col=0)

# Parse dates
news_df['date'] = pd.to_datetime(news_df['date'], utc=True, errors='coerce')
news_df = news_df.dropna(subset=['date'])
news_df['date'] = news_df['date'].dt.tz_localize(None)
news_df['trading_date'] = news_df['date'].dt.normalize()

print(f"✅ News data loaded! Shape: {news_df.shape}")
print(f"Columns: {list(news_df.columns)}")
print(f"Date range: {news_df['trading_date'].min().date()} to {news_df['trading_date'].max().date()}")

✅ News data loaded! Shape: (55987, 6)
Columns: ['headline', 'url', 'publisher', 'date', 'stock', 'trading_date']
Date range: 2011-04-28 to 2020-06-11


In [3]:
# Load all 5 stock CSV files
stocks = {
    'AAPL': '../data/raw/AAPL.csv',
    'AMZN': '../data/raw/AMZN.csv',
    'GOOG': '../data/raw/GOOG.csv',
    'META': '../data/raw/META.csv',
    'NVDA': '../data/raw/NVDA.csv'
}

stock_dfs = {}
for symbol, path in stocks.items():
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.set_index('Date')
    df.index = df.index.tz_localize(None)
    df = df.sort_index()
    df = df.dropna()
    # Compute daily return using Adj Close if available else Close
    df['Daily_Return'] = df['Close'].pct_change() * 100
    df = df.dropna()
    stock_dfs[symbol] = df
    print(f"✅ {symbol}: {len(df)} trading days | {df.index[0].date()} to {df.index[-1].date()}")

print(f"\n✅ All stock data loaded!")

✅ AAPL: 3773 trading days | 2009-01-05 to 2023-12-29
✅ AMZN: 3773 trading days | 2009-01-05 to 2023-12-29
✅ GOOG: 3773 trading days | 2009-01-05 to 2023-12-29
✅ META: 2922 trading days | 2012-05-21 to 2023-12-29
✅ NVDA: 3773 trading days | 2009-01-05 to 2023-12-29

✅ All stock data loaded!


In [4]:
# Filter news for our stocks
# Note: META was listed as FB before 2021
our_symbols = ['AAPL', 'AMZN', 'GOOG', 'GOOGL', 'NVDA', 'META', 'FB']

filtered_news = news_df[news_df['stock'].isin(our_symbols)].copy()

# Standardize symbols
filtered_news['stock'] = filtered_news['stock'].replace({
    'GOOGL': 'GOOG',
    'FB': 'META'
})

print(f"✅ Filtered news shape: {filtered_news.shape}")
print(f"\nArticles per stock:")
print(filtered_news['stock'].value_counts())
print(f"\nDate range: {filtered_news['trading_date'].min().date()} to {filtered_news['trading_date'].max().date()}")

✅ Filtered news shape: (60, 6)

Articles per stock:
stock
GOOG    20
AAPL    10
AMZN    10
META    10
NVDA    10
Name: count, dtype: int64

Date range: 2020-05-31 to 2020-06-10


In [5]:
# Initialize VADER
# We use VADER because:
# - Specifically designed for short text like news headlines
# - No training required
# - Returns compound score between -1 (negative) and +1 (positive)
# - Handles financial language well

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(headline):
    try:
        return analyzer.polarity_scores(str(headline))['compound']
    except:
        return 0.0

def classify_sentiment(score):
    if score > 0.05:
        return 'positive'
    elif score < -0.05:
        return 'negative'
    else:
        return 'neutral'

# Apply sentiment scoring
filtered_news['sentiment_score'] = filtered_news['headline'].apply(get_sentiment)
filtered_news['sentiment_label'] = filtered_news['sentiment_score'].apply(classify_sentiment)

print("✅ Sentiment analysis complete!")
print(f"\nSentiment score statistics:")
print(filtered_news['sentiment_score'].describe().round(4))
print(f"\nSentiment distribution:")
print(filtered_news['sentiment_label'].value_counts())

✅ Sentiment analysis complete!

Sentiment score statistics:
count    60.0000
mean      0.1526
std       0.3912
min      -0.8402
25%       0.0000
50%       0.0000
75%       0.5185
max       0.7296
Name: sentiment_score, dtype: float64

Sentiment distribution:
sentiment_label
positive    29
neutral     25
negative     6
Name: count, dtype: int64


In [6]:
# If multiple articles on same day for same stock
# → compute average daily sentiment score
daily_sentiment = filtered_news.groupby(
    ['stock', 'trading_date']
).agg(
    avg_sentiment=('sentiment_score', 'mean'),
    article_count=('headline', 'count'),
    sentiment_label=('sentiment_label', lambda x: x.mode()[0])
).reset_index()

print(f"✅ Daily sentiment aggregated!")
print(f"Shape: {daily_sentiment.shape}")
print(f"\nSample:")
print(daily_sentiment.head(10))

✅ Daily sentiment aggregated!
Shape: (18, 5)

Sample:
  stock trading_date  avg_sentiment  article_count sentiment_label
0  AAPL   2020-06-09       0.246900              4         neutral
1  AAPL   2020-06-10       0.198850              6        positive
2  AMZN   2020-06-09       0.077775              4        positive
3  AMZN   2020-06-10       0.391233              6        positive
4  GOOG   2020-06-04       0.000000              3         neutral
5  GOOG   2020-06-05      -0.411033              6        negative
6  GOOG   2020-06-07       0.000000              1         neutral
7  GOOG   2020-06-08       0.542300              2        positive
8  GOOG   2020-06-09       0.000000              3         neutral
9  GOOG   2020-06-10       0.319420              5        positive
